---
title: 'NeuroBridge: ECoG-to-Phoneme Decoding Pipeline'
labels: 'enhancement, model-architecture, bci'
---

## Project Goal
This project, **NeuroBridge**, aims to replicate and open-source the core architecture of a high-performance speech neuroprosthesis. The system translates intracranial electrocorticography (ECoG) signals directly into intelligible speech.

## Notebook Overview
This notebook demonstrates the end-to-end workflow using the `neurobridge` python package. It covers:
1.  **Configuration**: Setting up dataset and model parameters.
2.  **Mock Data Generation**: Simulating ECoG and phoneme label files to test the pipeline.
3.  **Data Loading**: Using `ECoGDatasetBuilder` to stream and preprocess data.
4.  **Model Training**: Training the `offline_decoder` (Bidirectional RNN).
5.  **Real-time Transition**: Building the `realtime_decoder` and transferring weights for inference.

In [ ]:
import os
import shutil
import numpy as np
import pandas as pd
import tensorflow as tf
from pathlib import Path
from scipy.io import savemat

# Import core components from the neurobridge package
from neurobridge.config import DatasetConfig, ModelConfig, TrainingConfig
from neurobridge.data_pipeline import PhonemeInventory, ECoGDatasetBuilder
from neurobridge.models import build_offline_decoder, build_realtime_decoder, initialize_realtime_from_offline

print("NeuroBridge libraries imported successfully.")

## 1. Configuration
We define the configuration for the dataset, model, and training process. 
Here we point to a local `mock_data_notebook` directory to store our simulated dataset.

In [ ]:
# Define configuration for the notebook environment
MOCK_DATA_DIR = Path("./mock_data_notebook")

if MOCK_DATA_DIR.exists():
    shutil.rmtree(MOCK_DATA_DIR)
MOCK_DATA_DIR.mkdir(parents=True)

dataset_cfg = DatasetConfig(
    ecog_dir=MOCK_DATA_DIR / "ecog",
    labels_dir=MOCK_DATA_DIR / "labels",
    mat_key='ecog_data',
    sampling_rate_hz=1000,
    target_rate_hz=100,
    window_duration_ms=500,
    stride_ms=100,
    num_features=64, # Reduced features for demonstration
    min_signal_duration_s=0.5
)

model_cfg = ModelConfig(
    architecture="rnn",
    dense_units=64,
    conv_filters=[32],
    rnn_units=[64],
    realtime_units=[64]
)

training_cfg = TrainingConfig(
    batch_size=8,
    max_epochs=5
)

print("Configuration initialized.")

## 2. Mock Data Generation
Since we do not have real patient data loaded, we generate random ECoG signals (`.mat` files) and corresponding phoneme label intervals (`.csv` files) on disk. This validates that the `ECoGDatasetBuilder` can correctly parse and align files from the filesystem.

In [ ]:
# Generate mock .mat and .csv files to simulate a real dataset
(dataset_cfg.ecog_dir).mkdir(exist_ok=True)
(dataset_cfg.labels_dir).mkdir(exist_ok=True)

num_samples = 20
timesteps_raw = 2000 # 2 seconds

print(f"Generating {num_samples} mock samples...")

for i in range(num_samples):
    # 1. Generate random ECoG data (white noise)
    raw_data = np.random.randn(timesteps_raw, dataset_cfg.num_features).astype(np.float32)
    savemat(dataset_cfg.ecog_dir / f"sample_{i}.mat", {dataset_cfg.mat_key: raw_data})

    # 2. Generate random phoneme labels
    intervals = []
    current_time = 0.1
    while current_time < (timesteps_raw / dataset_cfg.sampling_rate_hz) - 0.1:
        duration = np.random.uniform(0.1, 0.3)
        end_time = current_time + duration
        if end_time > (timesteps_raw / dataset_cfg.sampling_rate_hz):
            break
        ph = np.random.choice(dataset_cfg.phonemes[1:]) # Skip silence
        intervals.append({
            dataset_cfg.label_columns['start']: current_time, 
            dataset_cfg.label_columns['end']: end_time, 
            dataset_cfg.label_columns['label']: ph
        })
        current_time = end_time
        
    df = pd.DataFrame(intervals)
    df.to_csv(dataset_cfg.labels_dir / f"sample_{i}.csv", index=False)

print("Mock dataset created on disk.")

## 3. Pipeline Initialization
We instantiate the `ECoGDatasetBuilder` which handles:
- Scanning for valid file pairs.
- Loading and caching data.
- Signal preprocessing (Notch/Bandpass filters, Downsampling).
- Windowing and Label Alignment.
- Generating a `tf.data.Dataset`.

In [ ]:
# Initialize the data pipeline
inventory = PhonemeInventory(dataset_cfg.phonemes)
builder = ECoGDatasetBuilder(dataset_cfg, inventory, split="train", val_split=0.2)

# Create TensorFlow dataset
train_ds = builder.build_tf_dataset(batch_size=training_cfg.batch_size)

# Inspect one batch
for X, y in train_ds.take(1):
    print(f"Input shape: {X.shape}") # (batch, timesteps, features)
    print(f"Label shape: {y.shape}") # (batch, timesteps, num_classes)

## 4. Model Training
We use `build_offline_decoder` to construct the Neural Network (typically a Bidirectional LSTM or Transformer). This model is optimized for offline training where future context is available.

In [ ]:
# Build the offline decoder model
model = build_offline_decoder(dataset_cfg, model_cfg)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=training_cfg.learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# Train the model
print("\nStarting training...")
history = model.fit(train_ds, epochs=training_cfg.max_epochs)

# Save the model
model.save("neurobridge_offline_model.h5")
print("Offline model saved.")

## 5. Real-time Deployment
For live inference, we need a unidirectional model (no peeking into the future). We build a `realtime_decoder` with a compatible architecture and transfer the weights learned by the offline model. This enables immediate deployment without retraining from scratch.

In [ ]:
# Build the real-time equivalent model (unidirectional, stateful-ready logic)
rt_model = build_realtime_decoder(dataset_cfg, model_cfg)

# Transfer learned weights from the offline model
initialize_realtime_from_offline(model, rt_model)

rt_model.save("neurobridge_realtime_model.h5")
print("Real-time model built and weights transferred.")